# Customer Churn — Responsible ML Baseline

This notebook implements the exploratory baseline for the internship task **ML Problem Framing & Responsible Data Card**.

**Important:** The supplied dataset contains only 12 observations. Any model result here is for demonstration and responsible-problem-framing purposes, not production validation.


## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer-churn-training.csv")
df

## 2. Basic dataset inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Target and class balance

In [ ]:
target_counts = df["churned"].value_counts().sort_index()
target_pct = df["churned"].value_counts(normalize=True).sort_index() * 100

print("Counts:")
print(target_counts)
print("\nPercentages:")
print(target_pct.round(2))

print("\nMajority-class baseline accuracy:",
      round(target_counts.max() / len(df), 4))

## 4. Feature roles and descriptive statistics

`customer_id` is an identifier and is excluded from modeling.
`churned` is the target.


In [ ]:
numeric_features = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days",
]

categorical_features = ["plan_type"]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

display(df[numeric_features].describe().T)

print("\nChurn by plan type:")
display(df.groupby("plan_type")["churned"].agg(["count", "sum", "mean"]))

print("\nFeature means by churn outcome:")
display(df.groupby("churned")[numeric_features].mean().round(2))

## 5. Non-ML baseline

The baseline predicts the majority class for every observation. It is deliberately simple and gives us a reference point for deciding whether ML adds value.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

y = df["churned"]
majority_class = y.mode()[0]
baseline_pred = np.full(len(y), majority_class)

baseline_metrics = {
    "accuracy": accuracy_score(y, baseline_pred),
    "precision": precision_score(y, baseline_pred, zero_division=0),
    "recall": recall_score(y, baseline_pred, zero_division=0),
    "f1": f1_score(y, baseline_pred, zero_division=0),
}

print("Majority class:", majority_class)
print(pd.Series(baseline_metrics).round(4))
print("\nConfusion matrix:")
print(confusion_matrix(y, baseline_pred))

## 6. Exploratory ML baseline — Logistic Regression

Because the dataset is tiny, a single train/test split would make the evaluation highly unstable. We use **leave-one-out cross-validation (LOOCV)** only as an exploratory comparison.

The numerical features are standardized and `plan_type` is one-hot encoded.


In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

X = df[numeric_features + categorical_features]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

loo = LeaveOneOut()

ml_pred = cross_val_predict(model, X, y, cv=loo, method="predict")
ml_prob = cross_val_predict(model, X, y, cv=loo, method="predict_proba")[:, 1]

print("LOOCV predictions:")
display(pd.DataFrame({
    "customer_id": df["customer_id"],
    "actual": y,
    "predicted": ml_pred,
    "predicted_churn_probability": np.round(ml_prob, 3)
}))

## 7. Exploratory ML metrics

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

ml_metrics = {
    "accuracy": accuracy_score(y, ml_pred),
    "precision": precision_score(y, ml_pred, zero_division=0),
    "recall": recall_score(y, ml_pred, zero_division=0),
    "f1": f1_score(y, ml_pred, zero_division=0),
    "roc_auc": roc_auc_score(y, ml_prob),
    "brier_score": brier_score_loss(y, ml_prob),
}

print(pd.Series(ml_metrics).round(4))
print("\nConfusion matrix:")
print(confusion_matrix(y, ml_pred))

### Interpretation warning

The exploratory LOOCV result can be perfect on this tiny dataset. That does **not** mean the model will be perfect on future customers.

A particularly important warning sign is that every `Basic` observation is churned while every `Standard` and `Pro` observation is not churned. This can make `plan_type` an extremely strong shortcut in this sample. It may be a sampling artifact or a constructed relationship, and it should be validated on a larger, representative dataset.


## 8. Sensitivity check: exclude `plan_type`

This checks whether the numerical behavioral/account features still provide useful signal without the categorical plan feature.


In [ ]:
X_num = df[numeric_features]

numeric_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

num_pred = cross_val_predict(
    numeric_model, X_num, y, cv=loo, method="predict"
)

print(pd.Series({
    "accuracy": accuracy_score(y, num_pred),
    "precision": precision_score(y, num_pred, zero_division=0),
    "recall": recall_score(y, num_pred, zero_division=0),
    "f1": f1_score(y, num_pred, zero_division=0),
}).round(4))

print("\nConfusion matrix:")
print(confusion_matrix(y, num_pred))

## 9. Error analysis

For any future evaluation, inspect false positives and false negatives individually. The current sample is too small to estimate reliable operational error costs.

- **False positive:** unnecessary retention attention or incentive.
- **False negative:** missed retention opportunity.

The supplied materials do not specify monetary costs, so this project does not invent a cost ratio.


In [ ]:
errors = df.loc[ml_pred != y, ["customer_id"] + numeric_features + categorical_features + ["churned"]].copy()
errors["predicted"] = ml_pred[ml_pred != y]
errors

## 10. Responsible ML conclusions

1. The majority-class baseline provides a simple non-ML reference.
2. Logistic Regression can be demonstrated as an exploratory ML baseline.
3. The dataset has only 12 rows, so performance estimates are not reliable evidence for production.
4. Provenance, consent/licensing, sampling procedure, and feature timestamps are not supplied.
5. Human review, abstention for uncertain cases, monitoring, and rollback should be part of any future operational design.
6. Before deployment, collect substantially more representative data and define business costs for false positives and false negatives.
